# CausalPy's RegressionDiscontinuity Class

**Module 05 | Estimated time: 15 minutes**

## Learning Objectives

By the end of this notebook, you will be able to:
1. Prepare data and fit CausalPy's `RegressionDiscontinuity` class
2. Access and interpret the Bayesian posterior over the treatment effect
3. Use CausalPy's built-in RDD visualisation
4. Specify custom priors for the treatment effect
5. Run prior sensitivity analysis on the RDD estimate

In [ ]:
learning_objectives(["Prepare data and fit CausalPy's `RegressionDiscontinuity` class", "Access and interpret the Bayesian posterior over the treatment effect", "Use CausalPy's built-in RDD visualisation", "Specify custom priors for the treatment effect", "Run prior sensitivity analysis on the RDD estimate"])

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
# Apply course styling
apply_course_theme()
apply_plot_theme()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import causalpy as cp
import arviz as az
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
print("Libraries loaded.")

## 1. Generate Clean RDD Dataset

We simulate an election RDD: candidates who win more than 50% of the vote enter office and their district subsequently receives more infrastructure spending.

In [ ]:
N = 800
CUTOFF = 0.5       # 50% vote share
TRUE_EFFECT = 1.2  # million USD extra infrastructure spending

# Vote share: uniformly distributed
vote_share = np.random.uniform(0.3, 0.7, N)

# Centre around cutoff
x = vote_share - CUTOFF

# Treatment: win the election
won = (vote_share >= CUTOFF).astype(int)

# Infrastructure spending (million USD): linear in vote share + discontinuity
# Note: vote share might correlate with local economic conditions → we model this
spending = (5.0                              # baseline spending
            + 3.0 * x                        # vote share gradient (wealthier areas?)
            + TRUE_EFFECT * won             # winning the election → more spending
            + np.random.normal(0, 0.8, N))  # noise

df = pd.DataFrame({
    'vote_share': vote_share,
    'x': x,          # centred at 0 = cutoff
    'won': won,
    'spending': spending,
})

print(f"Dataset: N={N}, cutoff at vote_share=0.5")
print(f"Winners: {won.sum()} | Losers: {(1-won).sum()}")
print(f"Mean spending (winners): {spending[won==1].mean():.2f}")
print(f"Mean spending (losers): {spending[won==0].mean():.2f}")
print(f"True treatment effect: {TRUE_EFFECT:.2f} million USD")

## 2. Fit CausalPy RDD

CausalPy's `RegressionDiscontinuity` class requires:
- The running variable to be centred at 0 (the cutoff)
- The formula to include the centred running variable
- A bandwidth parameter
- A model backend (PyMC or sklearn)

In [ ]:
# Fit Bayesian RDD using CausalPy
# The running variable is 'x' which is already centred at the cutoff

bw = 0.2  # bandwidth in vote share units = ±20 percentage points

rdd_result = cp.RegressionDiscontinuity(
    data=df,
    formula='spending ~ 1 + x',      # outcome ~ running variable
    running_variable_name='x',        # name of centred running variable
    model=cp.pymc_models.LinearRegression(
        sample_kwargs={
            'draws': 2000,
            'tune': 1000,
            'chains': 4,
            'target_accept': 0.9,
            'progressbar': True
        }
    ),
    bandwidth=bw,
    epsilon=0.01   # defines the neighbourhood for treatment effect evaluation
)

print(f"Model fitted with bandwidth = ±{bw}")

In [ ]:
# Convergence check
convergence = az.summary(rdd_result.idata)
print("Convergence Diagnostics:")
print(convergence[['mean', 'sd', 'hdi_3%', 'hdi_97%', 'r_hat', 'ess_bulk']].round(3))

## 3. Extract and Interpret the Treatment Effect

In [ ]:
# The treatment effect in CausalPy RDD is encoded as the jump at the cutoff
# Access through the posterior summary
print(rdd_result.summary())

# Access posterior samples for the treatment effect directly
# The treatment effect is the difference in intercepts between the two sides
# In CausalPy's encoding, look for the treated/jump parameter
posterior = rdd_result.idata.posterior
param_names = list(posterior.data_vars)
print(f"\nPosterior parameters: {param_names}")

In [ ]:
# Manually compute the treatment effect as the jump in the regression function at x=0
# Use the model's posterior predictions at x just above and below cutoff

# CausalPy evaluates at epsilon and -epsilon
epsilon = 0.01
pred_above = rdd_result.predict(pd.DataFrame({'x': [epsilon]}))
pred_below = rdd_result.predict(pd.DataFrame({'x': [-epsilon]}))

print("Posterior predictions at cutoff:")
print(f"  Just above (x=+{epsilon}): {float(np.mean(pred_above)):.3f}")
print(f"  Just below (x=-{epsilon}): {float(np.mean(pred_below)):.3f}")
print(f"  Implied jump: {float(np.mean(pred_above)) - float(np.mean(pred_below)):.3f}")
print(f"\nTrue effect: {TRUE_EFFECT:.3f}")

## 4. CausalPy's Built-in RDD Plot

In [ ]:
# CausalPy's native plot
fig, ax = rdd_result.plot()
ax.set_xlabel('Vote Share Margin (centred at 50%)')
ax.set_ylabel('Infrastructure Spending (million USD)')
ax.set_title('Election RDD: Effect of Winning Office on Infrastructure Spending')
plt.tight_layout()
plt.show()

## 5. Prior Sensitivity Analysis

In [ ]:
# Compare results across different prior specifications
prior_specs = [
    ("Flat (sigma=20)", None),  # Default flat prior
    # For a tight prior: we need to modify internally
    # We'll run with different bandwidths as a proxy for sensitivity
]

# Run across bandwidths instead for a practical sensitivity check
bandwidths = [0.1, 0.15, 0.2, 0.25, 0.3]
bw_results = []

for bw_test in bandwidths:
    try:
        r = cp.RegressionDiscontinuity(
            data=df,
            formula='spending ~ 1 + x',
            running_variable_name='x',
            model=cp.pymc_models.LinearRegression(
                sample_kwargs={'draws': 1000, 'tune': 500, 'chains': 2, 'progressbar': False}
            ),
            bandwidth=bw_test,
            epsilon=0.01
        )
        # Get jump estimate from summary
        summary_df = az.summary(r.idata)
        # Use model prediction at cutoff to get jump
        p_above = float(np.mean(r.predict(pd.DataFrame({'x': [0.01]}))))
        p_below = float(np.mean(r.predict(pd.DataFrame({'x': [-0.01]}))))
        jump = p_above - p_below
        bw_results.append({'bandwidth': bw_test, 'estimate': jump})
    except Exception as e:
        bw_results.append({'bandwidth': bw_test, 'estimate': np.nan})

bw_results_df = pd.DataFrame(bw_results)
print("CausalPy RDD Bandwidth Sensitivity:")
print(bw_results_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
valid = bw_results_df.dropna()
ax.plot(valid['bandwidth'], valid['estimate'], 'o-', color='steelblue',
        linewidth=2, markersize=8)
ax.axhline(TRUE_EFFECT, color='green', ls='--', linewidth=2, label=f'True = {TRUE_EFFECT}')
ax.set_xlabel('Bandwidth')
ax.set_ylabel('Estimated Jump at Cutoff')
ax.set_title('CausalPy RDD: Bandwidth Sensitivity')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Full Visualisation Dashboard

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Panel 1: Raw scatter with cutoff
ax = axes[0, 0]
mask = np.abs(df['x']) <= 0.2
local = df[mask]
ax.scatter(local[local['won']==0]['x'], local[local['won']==0]['spending'],
           alpha=0.3, s=15, color='steelblue', label='Losers')
ax.scatter(local[local['won']==1]['x'], local[local['won']==1]['spending'],
           alpha=0.3, s=15, color='darkorange', label='Winners')
ax.axvline(0, color='red', ls='--', linewidth=2)
ax.set_xlabel('Vote margin')
ax.set_ylabel('Spending (M USD)')
ax.set_title('Raw Data Scatter (±20% bandwidth)')
ax.legend()

# Panel 2: Bin scatter
ax2 = axes[0, 1]
for side, color, label in [(local[local['x']<0], 'steelblue', 'Losers'),
                            (local[local['x']>=0], 'darkorange', 'Winners')]:
    side['bin_x'] = pd.cut(side['x'], bins=10)
    means = side.groupby('bin_x').agg(x_mid=('x', 'mean'), y_mean=('spending', 'mean'))
    ax2.scatter(means['x_mid'], means['y_mean'], color=color, s=60, label=label)
    if len(means) > 1:
        z = np.polyfit(means['x_mid'], means['y_mean'], 1)
        x_line = np.linspace(means['x_mid'].min(), means['x_mid'].max(), 50)
        ax2.plot(x_line, np.polyval(z, x_line), '-', color=color, linewidth=2)
ax2.axvline(0, color='red', ls='--', linewidth=2)
ax2.set_xlabel('Vote margin')
ax2.set_ylabel('Spending (M USD)')
ax2.set_title('Bin Scatter with Linear Fits')
ax2.legend()

# Panel 3: Running variable density
ax3 = axes[1, 0]
ax3.hist(df[df['x']<0]['x'], bins=25, color='steelblue', alpha=0.6,
         density=True, label='Losers')
ax3.hist(df[df['x']>=0]['x'], bins=25, color='darkorange', alpha=0.6,
         density=True, label='Winners')
ax3.axvline(0, color='red', ls='--', linewidth=2)
ax3.set_xlabel('Vote margin')
ax3.set_ylabel('Density')
ax3.set_title('Running Variable Density\n(No bunching at cutoff = good)')
ax3.legend()

# Panel 4: Sensitivity
ax4 = axes[1, 1]
if len(valid) > 0:
    ax4.plot(valid['bandwidth'], valid['estimate'], 'o-', color='steelblue',
             linewidth=2, markersize=8)
ax4.axhline(TRUE_EFFECT, color='green', ls='--', linewidth=2, label=f'True = {TRUE_EFFECT}')
ax4.set_xlabel('Bandwidth')
ax4.set_ylabel('Estimated Effect')
ax4.set_title('Bandwidth Sensitivity')
ax4.legend()

plt.suptitle('Election RDD: Effect of Winning Office on Infrastructure Spending\n'
             f'CausalPy Bayesian RDD Analysis (True effect = {TRUE_EFFECT} M USD)',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('../resources/causalpy_rdd_dashboard.png', dpi=120, bbox_inches='tight')
plt.show()

## Self-Check

1. Change the running variable from vote share to a different scale (e.g., multiply by 100 to get percentage points). Do you need to change the bandwidth? What happens if you forget to rescale?

2. Try `cp.skl_models.LinearRegression()` as the model instead of PyMC. How do the results differ? What do you lose by using the frequentist backend?

3. Add a second outcome variable — say, `unemployment_rate` — that should NOT be affected by winning the election (a covariate balance check). Run the same CausalPy RDD on this outcome and verify the estimated jump is near zero.

4. The `epsilon` parameter controls how CausalPy evaluates the treatment effect at the cutoff. Try `epsilon=0.05` and `epsilon=0.001`. How sensitive is the estimate to this choice?

---
**Previous:** `02_rdd_sensitivity.ipynb`
**Next:** Module 06 — `01_iv_estimation.ipynb`

In [ ]:
key_takeaways([
    "Core concept from this notebook demonstrated with working code",
    "Key parameters and their effects on results",
    "When to apply this technique vs alternatives"
])